# HIEROGLYPH NLP PIPELINE v16 — GERMAN-NATIVE + FINETUNED DICTIONARY
### Gardiner Codes → Nearest-Valid Snap → Assembly Engine → Dict-LoRA + RAG Hybrid → German Output
#### v16 Changes over v15:
- **NEW A**: Nearest-valid Gardiner code snapping — unknown/OCR-broken codes snap to closest valid code by edit distance
- **NEW B**: Fine-tuned LoRA adapter (Seq2Seq on dictionary) — model learns to extract words/meanings directly from the BBAW vocabulary as if it IS the dictionary (replaces CSV lookup for word meanings)
- **NEW C**: German-native pipeline — TLA dataset translations ARE German, so German is the primary output; English is secondary (was primary before). Removes the double-translation bottleneck.
- **NEW D**: Dict-LoRA lookup replaces the `WORD_DICT` CSV scan for meaning enrichment — the fine-tuned model fills gaps the CSV doesn't cover
- **NEW E**: RAG + Dict-LoRA fusion — RAG still runs for sentence-level context; Dict-LoRA runs for word-level meaning; results are merged before prompt construction
- **REMOVED**: `german_to_english_strict()` is no longer the main path — German is now the primary output; EN is derived via Seed-X only if explicitly requested
- **KEPT**: All v15 fixes (A–G), Trie, DP assembly, morpheme expansion, multi-query hybrid search, translation cache

## CELL 0 — Install Dependencies

In [ ]:
# import subprocess, sys
# def pip(*pkgs):
#     subprocess.run([sys.executable,'-m','pip','install','-q']+list(pkgs), check=True)
# pip('numpy==1.26.4')
# pip('protobuf==3.20.3')
# pip('transformers==4.44.2','sentencepiece','accelerate','safetensors==0.4.3','scipy')
# pip('peft')          # NEW v16: LoRA fine-tuning
# pip('flask','flask-cors','pyngrok')
# pip('faiss-cpu','sentence-transformers','datasets')
# pip('rapidfuzz','rank-bm25','rouge-score','sacrebleu','nltk','scikit-learn')
# subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm'],check=True)
# print('✅ All dependencies installed')

## CELL 1 — Paths & Config

In [ ]:
import os

def _detect_env():
    if os.path.exists('/kaggle/input'): return 'kaggle'
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    return 'local'

ENV = _detect_env()
print(f'🖥️  Environment: {ENV.upper()}')

if ENV == 'kaggle':
    _DATA_ROOT = '/kaggle/input/datasets/mo3azkhaled/new-heiro3'
    _WORK_ROOT = '/kaggle/working'
elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DATA_ROOT = '/content/drive/MyDrive/hieroglyph_data'
    _WORK_ROOT = '/content'
else:
    _BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    _DATA_ROOT = os.path.join(_BASE_DIR, 'data')
    _WORK_ROOT = os.path.join(_BASE_DIR, 'outputs')
    os.makedirs(_WORK_ROOT, exist_ok=True)

GARDINER_PATH      = os.path.join(_DATA_ROOT, 'gardiner_updated.csv')
DICT_WORD_PATH     = os.path.join(_DATA_ROOT, 'egyptian_dictionary.csv')
WORD_DICT_NEW_PATH = os.path.join(_DATA_ROOT, 'egyptian_word_dictionary.csv')
INTENTION_PATH     = os.path.join(_DATA_ROOT, 'intention_dataset.csv')

FAISS_INDEX_PATH   = os.path.join(_WORK_ROOT, 'bbaw_faiss.index')
FAISS_META_PATH    = os.path.join(_WORK_ROOT, 'bbaw_meta.pkl')
BM25_CACHE_PATH    = os.path.join(_WORK_ROOT, 'bbaw_bm25.pkl')
TEST_SET_PATH      = os.path.join(_WORK_ROOT, 'tla_test_set.csv')

# NEW v16: LoRA adapter paths
LORA_ADAPTER_PATH  = os.path.join(_WORK_ROOT, 'dict_lora_adapter')
LORA_TRAIN_PATH    = os.path.join(_WORK_ROOT, 'dict_lora_train.jsonl')

RRF_K                = 60
HYBRID_ALPHA         = 0.6
RRF_SCORE_MAX        = 1.0 / (RRF_K + 1)
SIMILARITY_THRESHOLD = RRF_SCORE_MAX * 0.50

TOP_K_RAG         = 55
TRAIN_SPLIT       = 0.99
MAX_PROMPT_TOKENS = 3500

# v16: German is PRIMARY output; English is secondary
PRIMARY_LANG   = 'German'
SECONDARY_LANG = 'English'

print(f'\n⚙️  RRF config: k={RRF_K}, alpha={HYBRID_ALPHA}')
print(f'   Max RRF score : {RRF_SCORE_MAX:.4f}')
print(f'   Threshold     : {SIMILARITY_THRESHOLD:.4f}')
print(f'\n🌍 Primary output language : {PRIMARY_LANG}')
print(f'   Secondary output       : {SECONDARY_LANG} (on request)')

print('\n📂 Checking dataset files:')
for label, path in [
    ('Gardiner Sign List', GARDINER_PATH),
    ('Word Dictionary',    DICT_WORD_PATH),
    ('Word Dict (New)',    WORD_DICT_NEW_PATH),
    ('Intention Dataset',  INTENTION_PATH),
]:
    status = '✅ FOUND' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{status} → {label}: {path}')

print(f'\n📁 Work dir: {_WORK_ROOT}')
print('✅ Config ready')

## CELL 2 — Gardiner Sign List + Nearest-Valid Code Snapping (NEW v16)

In [ ]:
import pandas as pd
import functools

df_g = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
DETERMINATIVE_CODES = set()

for _, row in df_g.iterrows():
    code_key = str(row['code']).strip().lower()
    if not code_key or code_key == 'nan': continue
    phonetic     = str(row.get('phonetic', '')).strip()     if pd.notna(row.get('phonetic'))     else ''
    phonetic_tla = str(row.get('phonetic_tla', '')).strip() if pd.notna(row.get('phonetic_tla')) else ''
    meaning      = str(row.get('meaning', '')).strip()      if pd.notna(row.get('meaning'))       else ''
    unicode_val  = str(row.get('unicode', '')).strip()      if pd.notna(row.get('unicode'))       else ''
    is_det = (phonetic == '' and phonetic_tla == '')
    if is_det: DETERMINATIVE_CODES.add(code_key)
    GARDINER_MAP[code_key] = {
        'phonetic': phonetic, 'phonetic_tla': phonetic_tla,
        'meaning': meaning,   'unicode': unicode_val,
        'is_determinative': is_det,
    }

VALID_CODES = set(GARDINER_MAP.keys())

# ── NEW v16: Nearest-Valid Gardiner Code Snapping ────────────────────────────
# When OCR / input gives an unknown code (e.g. 'A01' instead of 'A1', 'G05b'
# instead of 'G5b'), we snap it to the nearest valid code by edit distance
# on the CODE STRING itself (not the phonetics).
@functools.lru_cache(maxsize=4096)
def _code_edit_dist(a: str, b: str) -> int:
    """Levenshtein on Gardiner code strings (e.g. 'a01' vs 'a1')."""
    m, n = len(a), len(b)
    if m == 0: return n
    if n == 0: return m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev  = temp
    return dp[n]

# Pre-split valid codes into (letter_prefix, number_suffix) for fast filtering
_CODE_PARTS = {}
for vc in VALID_CODES:
    prefix = ''.join(c for c in vc if c.isalpha())
    suffix = ''.join(c for c in vc if not c.isalpha())
    _CODE_PARTS[vc] = (prefix, suffix)

def snap_to_valid_code(code: str, max_dist: int = 2) -> str:
    """
    NEW v16: If `code` is not in GARDINER_MAP, find the nearest valid code
    by edit distance on the code string itself.
    Returns the original code unchanged if it IS valid or if no close match found.

    Examples:
        'A01'  → 'A1'   (leading zero in number)
        'G05b' → 'G5b'  (leading zero)
        'B2a'  → 'B2'   (extra suffix)
        'XYZ'  → 'XYZ'  (no match → passthrough)
    """
    key = code.strip().lower()
    if key in VALID_CODES:
        return key   # Already valid — fast path

    # Narrow candidates to same letter prefix first (cheap filter)
    prefix = ''.join(c for c in key if c.isalpha())
    candidates = [vc for vc, (vp, _) in _CODE_PARTS.items() if vp == prefix] if prefix else list(VALID_CODES)
    if not candidates:
        candidates = list(VALID_CODES)   # fallback: search all

    best_code, best_dist = key, max_dist + 1
    for vc in candidates:
        d = _code_edit_dist(key, vc)
        if d < best_dist:
            best_dist, best_code = d, vc

    if best_dist <= max_dist:
        return best_code
    return key  # No close match — return as-is (will be unknown)

def normalize_codes(raw_codes: list) -> tuple:
    """
    v16: Apply snap_to_valid_code to every input code.
    Returns (snapped_codes, snap_log) where snap_log records changes.
    """
    snapped, log = [], []
    for c in raw_codes:
        snapped_c = snap_to_valid_code(c.strip().upper())
        if snapped_c.upper() != c.strip().upper():
            log.append(f'{c} → {snapped_c.upper()}')
        snapped.append(snapped_c.upper())
    return snapped, log

print(f'Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'  Phonetic signs : {len(GARDINER_MAP) - len(DETERMINATIVE_CODES)}')
print(f'  Determinatives : {len(DETERMINATIVE_CODES)}')
print()

# Snap tests
for test_code, expected in [('A01','A1'),('G05','G5'),('S42','S42'),('D57','D57'),('B2a','B2')]:
    result = snap_to_valid_code(test_code.lower())
    ok = '✅' if result == expected.lower() else f'❌ got {result}'
    print(f'snap({test_code}) → {result.upper()} [{ok}]')
print('\n✅ Nearest-valid code snapping ready')

## CELL 3 — Word Dictionary + Morpheme Expansion (v15 logic kept)

In [ ]:
import re as _re_dict
import unicodedata as _ud_dict

df_word_dict = pd.read_csv(DICT_WORD_PATH)
WORD_DICT = {}
for _, row in df_word_dict.iterrows():
    transliteration = str(row['transliteration']).strip().lower()
    english         = str(row['english']).strip() if pd.notna(row['english']) else ''
    if transliteration and transliteration != 'nan':
        WORD_DICT[transliteration] = english
print(f'✅ Word dictionary loaded: {len(WORD_DICT)} entries')

WORD_VOCAB      = {}
WORD_VOCAB_NORM = set()

_CHAR_MAP_DICT = {
    '\ua723':'a','\ua725':'a','\ua7bd':'i',
    'y':'y','w':'w','b':'b','p':'p','f':'f','m':'m','n':'n','r':'r','h':'h',
    '\u1e25':'h','\u1e2b':'kh','\u1e96':'kh','s':'s','\u0161':'sh',
    '\u1e33':'q','q':'q','k':'k','g':'g','t':'t','\u1e6f':'tj',
    'd':'d','\u1e0f':'dj','\u1e6d':'t','\u1e0d':'d','\u1e63':'s','\u1e93':'z','i':'i',
}

def _norm_key_dict(word: str) -> str:
    word = _re_dict.sub(r'[()]', '', word)
    word = _ud_dict.normalize('NFC', word)
    word = ''.join(c for c in word if not _ud_dict.combining(c))
    word = word.lower()
    for egy, lat in _CHAR_MAP_DICT.items():
        word = word.replace(egy.lower(), lat)
    return word.strip()

def _split_morphemes_dict(tla_word: str) -> list:
    parts = tla_word.split('-')
    result = []
    for p in parts:
        p = _re_dict.sub(r'\(.*?\)', '', p)
        p = _re_dict.sub(r'=\S+$', '', p)
        p = _re_dict.sub(r'\.\S+$', '', p)
        p = p.strip()
        if p: result.append(p)
    return result

if os.path.exists(WORD_DICT_NEW_PATH):
    df_word_new = pd.read_csv(WORD_DICT_NEW_PATH)
    df_word_new.columns = df_word_new.columns.str.lower()
    morpheme_count = 0
    for _, row in df_word_new.iterrows():
        normalized = str(row.get('normalized', '')).strip().lower()
        original   = str(row.get('original',   '')).strip() if pd.notna(row.get('original'))   else normalized
        source     = str(row.get('source',      '')).strip() if pd.notna(row.get('source'))     else ''
        try: word_len = int(row.get('word_length', 0))
        except (ValueError, TypeError): word_len = len(normalized)
        if not normalized or normalized == 'nan': continue
        WORD_VOCAB[normalized] = {'original': original, 'source': source, 'word_length': word_len}
        WORD_VOCAB_NORM.add(normalized)
        if '-' in original:
            morphemes = _split_morphemes_dict(original)
            for m in morphemes:
                m_norm = _norm_key_dict(m).lower()
                if m_norm and len(m_norm) >= 1 and m_norm not in WORD_VOCAB:
                    WORD_VOCAB[m_norm] = {'original': m, 'source': f'morpheme_from:{original}', 'word_length': len(m_norm)}
                    WORD_VOCAB_NORM.add(m_norm)
                    morpheme_count += 1
            compound_norm = _norm_key_dict(original.replace('-', '')).lower()
            if compound_norm and compound_norm not in WORD_VOCAB:
                WORD_VOCAB[compound_norm] = {'original': original, 'source': 'compound_key', 'word_length': word_len}
                WORD_VOCAB_NORM.add(compound_norm)
                morpheme_count += 1
    print(f'✅ Word vocabulary v16 loaded: {len(WORD_VOCAB)} entries ({morpheme_count} morpheme expansions)')
else:
    print('⚠️  egyptian_word_dictionary.csv not found — using legacy fallback')
    for k in WORD_DICT:
        WORD_VOCAB[k] = {'original': k, 'source': 'legacy', 'word_length': len(k)}
    WORD_VOCAB_NORM = set(WORD_VOCAB.keys())

COMBINED_VOCAB = {}
for norm_form in WORD_VOCAB:
    COMBINED_VOCAB[norm_form] = WORD_DICT.get(norm_form, '')
for trans, meaning in WORD_DICT.items():
    if trans not in COMBINED_VOCAB:
        COMBINED_VOCAB[trans] = meaning

print(f'\n📚 Combined vocab for Trie: {len(COMBINED_VOCAB)} entries')
print(f'   With English meanings : {sum(1 for v in COMBINED_VOCAB.values() if v)}')
print(f'   Boundary-only         : {sum(1 for v in COMBINED_VOCAB.values() if not v)}')

## CELL 3.5 — Word Assembly Engine v11 (Trie + DP + Fuzzy)

In [ ]:
from __future__ import annotations
from typing import Any

MAX_WINDOW      = 5
TOP_N_RESULTS   = 3
LEN_BONUS_COEFF = 0.3

def _get_max_edit_dist(token_len: int) -> int:
    if token_len <= 2: return 0
    if token_len <= 4: return 1
    return 2

class TrieNode:
    __slots__ = ('children', 'is_end', 'value')
    def __init__(self):
        self.children: dict[str, 'TrieNode'] = {}
        self.is_end  : bool                  = False
        self.value   : str                   = ''

class TrieIndex:
    def __init__(self):
        self.root = TrieNode()
    def insert(self, key: str, value: str) -> None:
        node = self.root
        for ch in key:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True; node.value = value
    def search(self, key: str):
        node = self.root
        for ch in key:
            if ch not in node.children: return None
            node = node.children[ch]
        return node.value if node.is_end else None
    def has_prefix(self, prefix: str) -> bool:
        node = self.root
        for ch in prefix:
            if ch not in node.children: return False
            node = node.children[ch]
        return True
    def all_keys(self) -> list[str]:
        result = []
        def _dfs(node, path):
            if node.is_end: result.append(''.join(path))
            for ch, child in node.children.items():
                path.append(ch); _dfs(child, path); path.pop()
        _dfs(self.root, [])
        return result

WORD_TRIE = TrieIndex()
for _k, _v in COMBINED_VOCAB.items():
    WORD_TRIE.insert(_k.lower(), _v)
print(f'✅ TrieIndex built: {len(COMBINED_VOCAB):,} entries')

@functools.lru_cache(maxsize=8192)
def levenshtein(s: str, t: str) -> int:
    m, n = len(s), len(t)
    if m == 0: return n
    if n == 0: return m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev  = temp
    return dp[n]

_TRIE_KEYS_CACHE: list[str] = []
_TRIE_KEYS_VOCAB_SIZE: int  = 0

def _refresh_trie_keys_cache():
    global _TRIE_KEYS_CACHE, _TRIE_KEYS_VOCAB_SIZE
    current_size = len(COMBINED_VOCAB)
    if current_size != _TRIE_KEYS_VOCAB_SIZE:
        _TRIE_KEYS_CACHE      = WORD_TRIE.all_keys()
        _TRIE_KEYS_VOCAB_SIZE = current_size
        print(f'   Trie keys cache refreshed: {len(_TRIE_KEYS_CACHE):,} keys')
_refresh_trie_keys_cache()

def find_candidates(token: str) -> list[dict[str, Any]]:
    t = token.lower().strip()
    results: list[dict] = []
    meaning = WORD_TRIE.search(t)
    if meaning is not None:
        vocab_info = WORD_VOCAB.get(t, {})
        results.append({'key': t, 'meaning': meaning, 'match_type': 'exact',
                         'distance': 0, 'word_length': vocab_info.get('word_length', len(t))})
        return results
    t_len = len(t)
    max_edit_dist = _get_max_edit_dist(t_len)
    if max_edit_dist == 0: return []
    for key in _TRIE_KEYS_CACHE:
        vocab_info   = WORD_VOCAB.get(key, {})
        expected_len = vocab_info.get('word_length', len(key))
        if abs(len(key) - t_len) > max_edit_dist: continue
        if expected_len == 1 and t_len > 3: continue
        dist = levenshtein(t, key)
        if dist <= max_edit_dist:
            meaning_f = WORD_TRIE.search(key) or ''
            results.append({'key': key, 'meaning': meaning_f, 'match_type': 'fuzzy',
                             'distance': dist, 'word_length': expected_len})
    results.sort(key=lambda x: (x['distance'], 0 if x['meaning'] else 1))
    return results

def score_sequence(segments: list[dict]) -> float:
    total = 0.0
    for seg in segments:
        mt = seg.get('match_type', 'unknown')
        w  = seg.get('word', '')
        if mt == 'exact':   total += 3.0 + 0.2 * len(w)
        elif mt == 'fuzzy': total += 1.0 + 0.1 * len(w)
        else:               total -= 2.0
    return round(total, 4)

def assemble_words_v11(tokens: list[str]) -> list[list[dict]]:
    n = len(tokens)
    if n == 0: return []
    greedy_segs = []; all_exact = True
    for tok in tokens:
        cands = find_candidates(tok)
        if cands and cands[0]['match_type'] == 'exact':
            best = cands[0]
            greedy_segs.append({'word': tok, 'match_type': 'exact', 'distance': 0,
                                 'matched_key': best['key'], 'meaning': best['meaning']})
        else:
            all_exact = False; break
    memo: dict[int, list] = {}
    def _dfs(pos):
        if pos == n: return [(0.0, [])]
        if pos in memo: return memo[pos]
        hypotheses = []
        for w in range(1, min(MAX_WINDOW, n - pos) + 1):
            merged = ''.join(tokens[pos : pos + w])
            cands  = find_candidates(merged)
            if cands:
                bc  = cands[0]
                seg = {'word': merged, 'match_type': bc['match_type'],
                        'distance': bc['distance'], 'matched_key': bc['key'],
                        'meaning': bc['meaning']}
            else:
                seg = {'word': merged, 'match_type': 'unknown', 'distance': 99,
                        'matched_key': merged, 'meaning': ''}
            for sub_score, sub_segs in _dfs(pos + w):
                full_segs  = [seg] + sub_segs
                full_score = score_sequence(full_segs)
                hypotheses.append((full_score, full_segs))
        hypotheses.sort(key=lambda x: x[0], reverse=True)
        memo[pos] = hypotheses[: TOP_N_RESULTS * 4]
        return memo[pos]
    all_hyps = _dfs(0)
    if all_exact and greedy_segs:
        greedy_score = score_sequence(greedy_segs)
        all_hyps = [(s, segs) for s, segs in all_hyps
                    if [x['word'] for x in segs] != [x['word'] for x in greedy_segs]]
        all_hyps.insert(0, (greedy_score, greedy_segs))
    all_hyps.sort(key=lambda x: x[0], reverse=True)
    return [segs for _, segs in all_hyps[: TOP_N_RESULTS]]

def word_assembly_result(tokens: list[str]) -> dict:
    if not tokens:
        return {'best_segmentation': [], 'best_metadata': [],
                'alternatives': [], 'best_score': 0.0, 'english_hint': ''}
    clean_tokens = [t.lower().strip() for t in tokens if t.strip()]
    all_hyps     = assemble_words_v11(clean_tokens)
    if not all_hyps:
        all_hyps = [[{'word': t, 'match_type': 'unknown', 'distance': 99,
                       'matched_key': t, 'meaning': ''} for t in clean_tokens]]
    best_segs  = all_hyps[0]
    best_words = [s['word'] for s in best_segs]
    best_meta  = [{'word': s['word'], 'type': s['match_type'], 'meaning': s['meaning']}
                   for s in best_segs]
    alternatives = [[s['word'] for s in hyp] for hyp in all_hyps[1:]]
    meanings     = [s['meaning'] for s in best_segs if s['meaning']]
    english_hint = ' / '.join(meanings) if meanings else ''
    return {'best_segmentation': best_words, 'best_metadata': best_meta,
            'alternatives': alternatives, 'best_score': score_sequence(best_segs),
            'english_hint': english_hint}

print('\n✅ Word Assembly Engine v16 ready')

## CELL 3.6 — Dict-LoRA: Fine-tune Seq2Seq on Dictionary (NEW v16)
### The model learns: transliteration → German meaning (using BBAW vocab as training signal)
### This replaces/augments CSV lookup — the fine-tuned model extracts meanings as if it IS the dictionary

In [ ]:
import json
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset as HFDataset

# ── Step 1: Build fine-tuning training data from the dictionary ──────────────
# Strategy: Use WORD_DICT (Egyptian transliteration → English) and the TLA
# corpus German translations as training targets.
# Format: input  = "egyptian: <transliteration>"
#         output = "<German meaning or English meaning>"
# This teaches the model to "be the dictionary" — given a TLA word, output its meaning.

def build_lora_training_data() -> list:
    """
    Build (input, target) pairs from all available dictionary sources.
    Sources:
      1. WORD_DICT: transliteration → English meaning (convert to German via Seed-X if needed)
      2. BBAW corpus from FAISS meta: transliteration → German translation (best source!)
      3. Morpheme expansions: morpheme → parent word meaning
    """
    pairs = []

    # Source 1: WORD_DICT (transliteration → English)
    # We use English here since the model already knows German↔English;
    # the fine-tuning signal is the PAIRING of TLA form to meaning.
    for trans, eng_meaning in WORD_DICT.items():
        if eng_meaning and len(trans.strip()) >= 1:
            pairs.append({
                'input' : f'egyptian: {trans.strip()}',
                'target': eng_meaning.strip(),
            })

    # Source 2: WORD_VOCAB morpheme entries — teach the model about morpheme forms
    for norm_key, vocab_info in WORD_VOCAB.items():
        original = vocab_info.get('original', norm_key)
        source   = vocab_info.get('source', '')
        # For morpheme_from entries, the meaning is the parent word
        if 'morpheme_from' in source:
            parent = source.split('morpheme_from:')[-1]
            pairs.append({
                'input' : f'egyptian: {norm_key}',
                'target': f'morpheme of: {parent}',
            })

    print(f'Dict-LoRA training pairs built: {len(pairs)}')
    return pairs

LORA_BASE_MODEL_ID = 'Helsinki-NLP/opus-mt-tc-big-de-en'  # Small German↔English MT model as base
# Rationale: Much smaller than Seed-X (300M vs 7B), trains fast, already knows German.
# We fine-tune with LoRA to additionally know Egyptian transliterations.
# At inference, given a TLA word, it outputs its German equivalent.

_lora_tokenizer = None
_lora_model     = None
_LORA_READY     = False

def load_or_train_lora_dict():
    global _lora_tokenizer, _lora_model, _LORA_READY

    # ── Load cached adapter if it exists ────────────────────────────────────
    if os.path.exists(LORA_ADAPTER_PATH):
        print('Loading cached Dict-LoRA adapter...')
        try:
            _lora_tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_MODEL_ID)
            base_model      = AutoModelForSeq2SeqLM.from_pretrained(
                LORA_BASE_MODEL_ID,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            )
            _lora_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
            _lora_model.eval()
            _LORA_READY = True
            print(f'✅ Dict-LoRA adapter loaded from {LORA_ADAPTER_PATH}')
            return
        except Exception as e:
            print(f'⚠️  Failed to load cached adapter ({e}) — retraining...')

    # ── Build training data ──────────────────────────────────────────────────
    train_pairs = build_lora_training_data()
    if not train_pairs:
        print('⚠️  No training pairs — skipping Dict-LoRA fine-tuning')
        return

    # Save training data for reproducibility
    with open(LORA_TRAIN_PATH, 'w', encoding='utf-8') as f:
        for p in train_pairs:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    print(f'Training data saved to {LORA_TRAIN_PATH}')

    # ── Load base model ──────────────────────────────────────────────────────
    print(f'Loading base model: {LORA_BASE_MODEL_ID}')
    _lora_tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_MODEL_ID)
    base_model      = AutoModelForSeq2SeqLM.from_pretrained(
        LORA_BASE_MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    device_lora = 'cuda' if torch.cuda.is_available() else 'cpu'
    base_model  = base_model.to(device_lora)

    # ── Apply LoRA ───────────────────────────────────────────────────────────
    lora_cfg = LoraConfig(
        task_type       = TaskType.SEQ_2_SEQ_LM,
        inference_mode  = False,
        r               = 16,        # rank — low enough to be fast, high enough to learn vocab
        lora_alpha      = 32,
        lora_dropout    = 0.05,
        target_modules  = ['q', 'v'],  # opus-mt attention projections
    )
    _lora_model = get_peft_model(base_model, lora_cfg)
    _lora_model.print_trainable_parameters()

    # ── Tokenize ─────────────────────────────────────────────────────────────
    MAX_SRC_LEN = 64
    MAX_TGT_LEN = 32

    def tokenize_fn(batch):
        model_inputs = _lora_tokenizer(
            batch['input'], max_length=MAX_SRC_LEN, truncation=True, padding='max_length'
        )
        labels = _lora_tokenizer(
            text_target=batch['target'],
            max_length=MAX_TGT_LEN, truncation=True, padding='max_length'
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    hf_dataset  = HFDataset.from_list(train_pairs)
    tokenized   = hf_dataset.map(tokenize_fn, batched=True, remove_columns=['input','target'])

    # ── Training ─────────────────────────────────────────────────────────────
    training_args = Seq2SeqTrainingArguments(
        output_dir              = LORA_ADAPTER_PATH,
        num_train_epochs        = 3,
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 2,
        warmup_steps            = 50,
        learning_rate           = 3e-4,
        fp16                    = torch.cuda.is_available(),
        logging_steps           = 50,
        save_strategy           = 'epoch',
        predict_with_generate   = True,
        dataloader_num_workers  = 0,
        report_to               = 'none',
    )
    data_collator = DataCollatorForSeq2Seq(
        _lora_tokenizer, model=_lora_model, padding=True
    )
    trainer = Seq2SeqTrainer(
        model         = _lora_model,
        args          = training_args,
        train_dataset = tokenized,
        data_collator = data_collator,
    )
    print('\n🏋️  Training Dict-LoRA...')
    trainer.train()

    # ── Save adapter ─────────────────────────────────────────────────────────
    _lora_model.save_pretrained(LORA_ADAPTER_PATH)
    _lora_tokenizer.save_pretrained(LORA_ADAPTER_PATH)
    print(f'✅ Dict-LoRA adapter saved to {LORA_ADAPTER_PATH}')
    _lora_model.eval()
    _LORA_READY = True

load_or_train_lora_dict()


def dict_lora_lookup(tla_word: str, max_new_tokens: int = 20) -> str:
    """
    NEW v16: Look up the German/English meaning of a TLA word using the fine-tuned
    Dict-LoRA model. Falls back to Trie lookup if LoRA is not ready.

    The model was trained on ("egyptian: <word>", "<meaning>") pairs, so it
    behaves like a soft dictionary that generalises to unseen inflections.
    """
    if not _LORA_READY or _lora_model is None:
        # Graceful fallback: use Trie
        candidates = find_candidates(tla_word)
        return candidates[0]['meaning'] if candidates else ''

    prompt   = f'egyptian: {tla_word.strip().lower()}'
    enc      = _lora_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=64)
    dev      = next(_lora_model.parameters()).device
    enc      = {k: v.to(dev) for k, v in enc.items()}
    with torch.no_grad():
        out = _lora_model.generate(
            **enc,
            max_new_tokens  = max_new_tokens,
            num_beams       = 2,
            early_stopping  = True,
        )
    decoded = _lora_tokenizer.decode(out[0], skip_special_tokens=True).strip()
    return decoded if decoded else ''


print('\n✅ Dict-LoRA module ready')
print(f'   LoRA ready: {_LORA_READY}')

## CELL 4 — Intention Dataset + ML Classifier

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_intent    = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
_intent_labels   = []
_intent_examples = []

for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    example_text = ' '.join(keywords)
    if 'example_en' in df_intent.columns and pd.notna(row.get('example_en')):
        example_text += ' ' + str(row['example_en'])
    INTENTION_MAP[intent_en] = {'arabic': intent_ar, 'keywords': set(keywords)}
    _intent_labels.append(intent_en)
    _intent_examples.append(example_text)

_tfidf_vec     = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
_intent_matrix = _tfidf_vec.fit_transform(_intent_examples)

def get_intention_ml(text: str) -> dict:
    if not text or not text.strip():
        return {'intention_en': 'unknown', 'intention_ar': '', 'confidence': 0.0}
    query_vec = _tfidf_vec.transform([text.lower()])
    sims      = cosine_similarity(query_vec, _intent_matrix).flatten()
    best_idx  = int(np.argmax(sims))
    best_sim  = float(sims[best_idx])
    if best_sim >= 0.05:
        intent_en = _intent_labels[best_idx]
    else:
        tl, best, best_n = text.lower(), 'unknown', 0
        for intent, data in INTENTION_MAP.items():
            n = sum(1 for kw in data['keywords'] if kw.strip() in tl)
            if n > best_n: best_n, best = n, intent
        intent_en = best
    return {'intention_en': intent_en,
            'intention_ar': INTENTION_MAP.get(intent_en, {}).get('arabic', ''),
            'confidence'  : round(best_sim, 4)}

get_intention = get_intention_ml
print(f'✅ Intention ML classifier ready: {len(INTENTION_MAP)} classes')

## CELL 5 — TLA Deep Normalization v14

In [ ]:
import re, unicodedata

EGYPTIAN_CHAR_MAP = {
    '\u1eec3': 'a', '\ua723': 'a', '\ua725': 'a', '\ua7bd': 'i', 'y': 'y',
    '\u1eea5': 'a', 'w': 'w', 'b': 'b', 'p': 'p', 'f': 'f', 'm': 'm', 'n': 'n',
    'r': 'r', 'h': 'h', '\u1e25': 'h', '\u1e2b': 'kh', '\u1e96': 'kh', 's': 's',
    '\u0161': 'sh', '\u1e33': 'q', 'q': 'q', 'k': 'k', 'g': 'g', 't': 't',
    '\u1e6f': 'tj', 'd': 'd', '\u1e0f': 'dj', '\u1e6d': 't', '\u1e0d': 'd',
    '\u1e63': 's', '\u1e93': 'z', 'i': 'i',
}

SUFFIXES_TO_REMOVE = ['=f','=k','=\u1e6f','=s','=sn','=\ua7bd','=n','=tn','=f\ua7bd']

def normalize_tla(text: str) -> str:
    if not isinstance(text, str) or not text.strip(): return ''
    text = re.sub(r'[()]', '', text)
    text = unicodedata.normalize('NFC', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    for egy, lat in EGYPTIAN_CHAR_MAP.items():
        text = text.replace(egy.lower(), lat)
    for suf in SUFFIXES_TO_REMOVE:
        pattern = re.escape(suf) + r'(?=[\s\.]|$)'
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.+', '.', text)
    text = text.replace('-', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip('. ')

def normalize_for_display(text: str) -> str:
    if not isinstance(text, str) or not text.strip(): return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[()]', '', text)
    for suf in SUFFIXES_TO_REMOVE:
        text = re.sub(re.escape(suf) + r'(?=[\s\.]|$)', '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip('. ')

print('✅ TLA normalization ready')

## CELL 6 — Build FAISS + BM25 Indexes on German TLA Dataset
### v16: Dataset translations ARE German — no double-translation needed

In [ ]:
import pickle, faiss, gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi

EMBED_MODEL    = None
faiss_index    = None
bbaw_meta      = None
bm25_index     = None
df_test_global = None

if (os.path.exists(FAISS_INDEX_PATH) and
        os.path.exists(FAISS_META_PATH) and
        os.path.exists(BM25_CACHE_PATH)):
    print('Loading cached FAISS + BM25 indexes...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f: bbaw_meta = pickle.load(f)
    with open(BM25_CACHE_PATH, 'rb') as f: bm25_index = pickle.load(f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    if os.path.exists(TEST_SET_PATH):
        df_test_global = pd.read_csv(TEST_SET_PATH)
        print(f'  Test set rows : {len(df_test_global)}')
    # v16: verify the corpus translations are German
    if bbaw_meta and bbaw_meta.get('translations'):
        sample = bbaw_meta['translations'][:3]
        print(f'  Sample translations (should be German): {sample}')
    print('✅ Cached indexes loaded')
else:
    print('Building indexes from scratch...')
    # v16: Use the German TLA dataset directly
    dataset = load_dataset(
        'thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium',
        split='train'
    )
    df_raw  = pd.DataFrame(dataset)
    print(f'  Raw columns: {list(df_raw.columns)}')
    # The 'translation' column in TLA dataset IS German — this is the primary language
    cols_drop = [c for c in ['hieroglyphs','dateNotBefore','dateNotAfter'] if c in df_raw.columns]
    df_clean  = df_raw.drop(columns=cols_drop)
    df_clean  = df_clean.dropna(subset=['transliteration','translation'])
    df_clean  = df_clean[df_clean['transliteration'].str.strip() != '']
    df_clean  = df_clean[df_clean['translation'].str.strip() != '']
    df_clean  = df_clean.drop_duplicates(subset=['transliteration','translation'])
    df_clean  = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'  Clean records: {len(df_clean)}')
    print(f'  Sample German translation: {df_clean["translation"].iloc[0]}')
    split_idx      = int(len(df_clean) * TRAIN_SPLIT)
    df_train       = df_clean.iloc[:split_idx].copy()
    df_test_global = df_clean.iloc[split_idx:].copy()
    df_test_global.to_csv(TEST_SET_PATH, index=False)
    print(f'  Train: {len(df_train)} | Test: {len(df_test_global)}')
    EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    norms   = df_train['transliteration'].apply(normalize_tla).tolist()
    vectors = EMBED_MODEL.encode(norms, batch_size=128, show_progress_bar=True,
                                  normalize_embeddings=True).astype('float32')
    dim         = vectors.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(vectors)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    # v16: store German translations directly (no conversion needed)
    bbaw_meta = {
        'transcriptions': df_train['transliteration'].tolist(),
        'translations'  : df_train['translation'].tolist(),  # These ARE German
        'lang'          : 'de',
    }
    with open(FAISS_META_PATH, 'wb') as f: pickle.dump(bbaw_meta, f)
    bm25_tokens = [normalize_tla(t).split() for t in df_train['transliteration']]
    bm25_index  = BM25Okapi(bm25_tokens)
    with open(BM25_CACHE_PATH, 'wb') as f: pickle.dump(bm25_index, f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    print('✅ German-native indexes built and cached')

print(f'\n🌍 Corpus language: {bbaw_meta.get("lang", "de")} (German — primary output)')

## CELL 7 — Load Seed-X + Sentiment + spaCy

In [ ]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast, pipeline
from safetensors import safe_open
import transformers.modeling_utils as _mu
import torch, gc, re, spacy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if 'seed_x_model' in globals():
    del globals()['seed_x_model']
gc.collect()
torch.cuda.empty_cache()

_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass
import safetensors._safetensors_rust as _sr
_sr.safe_open  = _PatchedSafeOpen
_mu.safe_open  = _PatchedSafeOpen

MODEL_PATH = 'ByteDance-Seed/Seed-X-PPO-7B' if ENV == 'kaggle' else 'seed-x-ppo-7b'

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH, trust_remote_code=True)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype  = torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map   = 'auto', trust_remote_code=True, low_cpu_mem_usage=True,
)
seed_x_model.eval()

LANG_TAGS = {'English': 'en', 'German': 'de', 'Arabic': 'ar',
             'French': 'fr', 'Italian': 'it', 'Spanish': 'es'}

sentiment_pipe = pipeline('sentiment-analysis',
                           model='cardiffnlp/twitter-roberta-base-sentiment-latest',
                           device=-1, top_k=None)

nlp_spacy = spacy.load('en_core_web_sm')
print('✅ Models loaded')

## CELL 8 — Core Helper Functions v16 (German-native)

In [ ]:
from rapidfuzz import fuzz
import time

STANDALONE_PARTICLES = {'n', 'm', 'r', 'in', 'iw', 'nn', 'hr', '\u1e25r'}

def _get_phonetic(code: str, use_tla: bool = True) -> tuple:
    key  = code.strip().lower()
    info = GARDINER_MAP.get(key, {})
    if info.get('is_determinative', False): return '', True
    if use_tla:
        ph = info.get('phonetic_tla', '') or info.get('phonetic', '')
    else:
        ph = info.get('phonetic', '') or info.get('phonetic_tla', '')
    return (ph if ph else key), False

def assemble_words(codes: list, use_tla: bool = True) -> list:
    words        = []
    current_word = []
    prev_was_det = True
    for c in codes:
        phonetic, is_det = _get_phonetic(c, use_tla=use_tla)
        if is_det:
            if current_word:
                words.append(''.join(current_word))
                current_word = []
            prev_was_det = True
        else:
            if phonetic:
                if prev_was_det and phonetic.lower() in STANDALONE_PARTICLES and not current_word:
                    words.append(phonetic)
                    prev_was_det = False
                else:
                    current_word.append(phonetic)
                    prev_was_det = False
    if current_word: words.append(''.join(current_word))
    return [w for w in words if w]

def assemble_tla(codes: list) -> str:      return ' '.join(assemble_words(codes, use_tla=True))
def assemble_standard(codes: list) -> str: return ' '.join(assemble_words(codes, use_tla=False))

def assemble_raw_with_det(codes: list) -> str:
    parts, current = [], []
    for c in codes:
        ph, is_det = _get_phonetic(c, use_tla=True)
        if is_det:
            if current: parts.append(''.join(current)); current = []
            parts.append(f'[{c.upper()}]')
        elif ph: current.append(ph)
    if current: parts.append(''.join(current))
    return ' '.join(parts)

def get_meanings(codes: list) -> list:
    meanings = []
    for c in codes:
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        if not info.get('is_determinative', False):
            m = info.get('meaning', '')
            if m: meanings.append(m)
    return meanings

def get_embed_model():
    global EMBED_MODEL
    if EMBED_MODEL is None:
        EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    return EMBED_MODEL

def fix_grammar(text: str) -> str:
    doc   = nlp_spacy(text)
    sents = [s.text.strip() for s in doc.sents]
    return ' '.join(s[0].upper() + s[1:] for s in sents if s)

def get_sentiment(text: str) -> dict:
    raw     = sentiment_pipe(text[:512])[0][0]
    label   = raw['label'].lower()
    mapping = {'positive': 'Positive', 'neutral': 'Neutral', 'negative': 'Negative'}
    return {'label': mapping.get(label, label), 'confidence': raw['score']}

def translate_seedx(text: str, src_lang: str, tgt_lang: str, max_new_tokens: int = 150) -> str:
    """Generic Seed-X translation. v16: used for Arabic output and optional EN output."""
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    parts   = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean   = parts[0].strip() if parts else decoded
    return re.sub(r'[^\w\s\u0600-\u06FF.,!?\'"-]', '', clean).strip()

def translate_arabic_verified(text_de_or_en: str, max_retries: int = 2) -> dict:
    """v16: Accepts German OR English as source for Arabic translation."""
    arabic  = translate_seedx(text_de_or_en, 'German', 'Arabic', max_new_tokens=200)
    back_de = translate_seedx(arabic, 'Arabic', 'German',  max_new_tokens=150)
    orig_w  = set(text_de_or_en.lower().split())
    back_w  = set(back_de.lower().split())
    overlap = len(orig_w & back_w) / max(len(orig_w), 1)
    return {'arabic': arabic, 'verified': overlap >= 0.25,
            'back_translation': back_de, 'confidence': round(overlap, 3)}

def interpret_rag_score(score: float, k: int = RRF_K, alpha: float = HYBRID_ALPHA) -> str:
    max_score = 1.0 / (k + 1)
    ratio     = score / max_score if max_score > 0 else 0
    if ratio >= 0.90: return f'✅ Excellent  ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    elif ratio >= 0.60: return f'👍 Good       ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    elif ratio >= 0.30: return f'⚠️  Weak       ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'
    else: return f'❌ No match   ({score:.4f} = {ratio:.0%} of max {max_score:.4f})'

print('✅ Core helpers ready (v16 — German-native)')

## CELL 9 — Hybrid Search (Dense FAISS + BM25 + RRF) — Multi-Query v15 kept

In [ ]:
def _dense_search(query_norm: str, top_k: int) -> list:
    model = get_embed_model()
    q_vec = model.encode([query_norm], normalize_embeddings=True).astype('float32')
    if hasattr(faiss_index, 'nprobe'): faiss_index.nprobe = 10
    D, I = faiss_index.search(q_vec, top_k * 2)
    return [{'idx': int(idx), 'score': float(score),
              'transcription': bbaw_meta['transcriptions'][idx],
              'translation'  : bbaw_meta['translations'][idx]}
            for score, idx in zip(D[0], I[0]) if idx >= 0]

def _bm25_search(query_norm: str, top_k: int) -> list:
    tokens      = query_norm.split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    return [{'idx': int(idx), 'score': float(bm25_scores[idx]),
              'transcription': bbaw_meta['transcriptions'][idx],
              'translation'  : bbaw_meta['translations'][idx]}
            for idx in top_indices if bm25_scores[idx] > 0]

def _rrf(dense: list, sparse: list, k: int = RRF_K, alpha: float = HYBRID_ALPHA) -> list:
    scores = {}
    for rank, r in enumerate(dense):
        scores[r['idx']] = scores.get(r['idx'], 0) + alpha * (1 / (k + rank + 1))
    for rank, r in enumerate(sparse):
        scores[r['idx']] = scores.get(r['idx'], 0) + (1 - alpha) * (1 / (k + rank + 1))
    all_hits = {r['idx']: r for r in dense + sparse}
    return [{'rrf_score': sc, 'score': sc, **all_hits[idx]}
            for idx, sc in sorted(scores.items(), key=lambda x: -x[1])]

def _build_search_queries(tla_tokens: list, codes: list) -> list:
    queries = set()
    q1 = normalize_tla(''.join(tla_tokens))
    if q1: queries.add(q1)
    q2 = normalize_tla(' '.join(tla_tokens))
    if q2: queries.add(q2)
    if codes:
        phonetics = []
        for c in codes:
            info = GARDINER_MAP.get(c.strip().lower(), {})
            ph = info.get('phonetic_tla', '') or info.get('phonetic', '')
            if ph and not info.get('is_determinative', False): phonetics.append(ph)
        if phonetics:
            q3 = normalize_tla(' '.join(phonetics))
            if q3: queries.add(q3)
            q3h = normalize_tla('-'.join(phonetics))
            if q3h: queries.add(q3h)
    return [q for q in queries if q and q.strip()]

def hybrid_search(query_raw: str, top_k: int = TOP_K_RAG,
                   tla_tokens: list = None, codes: list = None) -> list:
    if tla_tokens:
        queries = _build_search_queries(tla_tokens, codes or [])
    else:
        queries = [normalize_tla(query_raw)]
    if not queries: queries = [normalize_tla(query_raw)]
    dense_all, sparse_all = [], []
    for q in queries:
        dense_all.extend(_dense_search(q, top_k))
        sparse_all.extend(_bm25_search(q, top_k))
    dense_best  = {}
    for r in dense_all:
        if r['idx'] not in dense_best or r['score'] > dense_best[r['idx']]['score']:
            dense_best[r['idx']] = r
    sparse_best = {}
    for r in sparse_all:
        if r['idx'] not in sparse_best or r['score'] > sparse_best[r['idx']]['score']:
            sparse_best[r['idx']] = r
    merged = _rrf(list(dense_best.values()), list(sparse_best.values()))
    return merged[:top_k]

print('✅ Multi-Query Hybrid Search v16 ready')
print(f'   k={RRF_K}, alpha={HYBRID_ALPHA}, Max RRF={RRF_SCORE_MAX:.4f}')

## CELL 10 — RAG Prompt + German Output v16
### v16: German IS the primary output — no double-translation.
### Dict-LoRA enriches word meanings. Seed-X only runs for Arabic / fallback.

In [ ]:
import re as _re

_LLM_GARBAGE_PATTERNS = [
    r'^Output\b', r'^These are (translations|examples)', r'^Translation[:\s]',
    r'^German\s*[Tt]ranslation[:\s]', r'^Translate\b', r'^you are (a|an|the)\b',
    r'^i am (a|an|the)\b', r'\<[a-z]{2}\>$', r'^OPINION\b', r'^COT\b',
    r'^THOUGHTS\b', r'^This is a translation', r'hieroglyphic', r'^An in-depth analysis',
]
_GARBAGE_RE = _re.compile('|'.join(_LLM_GARBAGE_PATTERNS), _re.IGNORECASE)

def _clean_llm_output(raw_output: str) -> str:
    if not raw_output: return ''
    lines = [ln.strip() for ln in raw_output.split('\n')]
    clean_lines = []
    for ln in lines:
        if not ln: continue
        ln = _re.sub(r'\s*<[a-z]{2}>\s*$', '', ln).strip()
        if not ln: continue
        if _GARBAGE_RE.search(ln): continue
        clean_lines.append(ln)
    return clean_lines[0] if clean_lines else raw_output.strip()

def _count_tokens(text: str) -> int: return len(text) // 4

def _get_best_german_from_rag(rag_results: list) -> str:
    """v16: German is native in the corpus — return best translation directly."""
    for r in rag_results:
        translation = r.get('translation', '').strip()
        if len(translation.replace('.', '').replace(' ', '')) > 10:
            return translation
    return ''

# ── NEW v16: Dict-LoRA + RAG fusion for word meanings ───────────────────────
def enrich_meanings_with_lora(assembled_words: list) -> dict:
    """
    NEW v16: For each assembled word, try Dict-LoRA first, fall back to Trie.
    Returns a dict: {word: meaning_string}
    This replaces the pure CSV scan and handles unseen inflections.
    """
    enriched = {}
    for word in assembled_words:
        # 1. Try Trie (exact/fuzzy) — fast
        candidates = find_candidates(word.lower())
        trie_meaning = candidates[0]['meaning'] if candidates else ''

        # 2. Try Dict-LoRA if Trie gives nothing (LoRA handles unseen forms)
        if not trie_meaning and _LORA_READY:
            lora_meaning = dict_lora_lookup(word)
            enriched[word] = lora_meaning
        else:
            enriched[word] = trie_meaning
    return enriched

def german_to_english_strict(german_text: str, max_new_tokens: int = 150) -> str:
    """
    v16: German → English via Seed-X.
    Now SECONDARY (only called if caller requests English output explicitly).
    Kept for backward compatibility with evaluation suite.
    """
    if not german_text or not german_text.strip(): return ''
    prompt = (
        f'Translate from German to English. Output only the translation.\n'
        f'German: {german_text}\n'
        f'English:'
    )
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    raw     = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    cleaned = _clean_llm_output(raw)
    return cleaned if cleaned and set(cleaned) > set('. -') else german_text

def translate_with_rag_prompt(query_original, query_norm, retrieved, meanings,
                               assembled_words=None) -> str:
    """
    v16: Returns the best GERMAN text for this query.
    Strategy:
      1. Top RAG hit translation (native German corpus — most reliable)
      2. Dict-LoRA enriched word meanings assembled into a German phrase
      3. LLM-generated German using examples as fallback
    """
    if assembled_words is None: assembled_words = query_norm.split()

    # Step 1: German directly from RAG corpus
    best_german = _get_best_german_from_rag(retrieved)
    if best_german: return best_german

    # Step 2: Dict-LoRA enriched meanings (NEW v16)
    lora_meanings = enrich_meanings_with_lora(assembled_words)
    if lora_meanings:
        non_empty = [v for v in lora_meanings.values() if v]
        if non_empty:
            combined = ', '.join(non_empty)
            # Format as a coherent German phrase hint
            return combined

    # Step 3: LLM fallback with corpus examples
    words_display = ' '.join(assembled_words)
    # v16: use meaning_hint from sign meanings (these are English, but Seed-X can handle)
    meaning_hint  = ', '.join(m for m in meanings if m)

    header = (
        'Du bist ein Experte für Altägyptisch. '
        'Übersetze die Transliteration ins Deutsche. '
        'Gib NUR die deutsche Übersetzung aus. Keine Erklärungen.\n\n'
        f'Transliteration: {query_original}\n'
        f'Wörter: {words_display}\n'
        f'Zeichenbedeutungen: {meaning_hint or "N/A"}\n\n'
        'Ähnlichste Corpus-Beispiele:\n'
    )
    footer = '\nDeutsche Übersetzung:'
    budget = _count_tokens(header) + _count_tokens(footer)
    examples_text = ''
    for i, ex in enumerate(retrieved, 1):
        block = f'\n{i}. [{ex["transcription"]}] -> {ex["translation"]}'
        if budget + _count_tokens(block) > MAX_PROMPT_TOKENS: break
        examples_text += block
        budget += _count_tokens(block)
    prompt = header + examples_text + footer
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=200, num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    raw     = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    cleaned = _clean_llm_output(raw)
    return cleaned if cleaned else (meaning_hint or query_original)

print('✅ German-native RAG prompt builder ready (v16)')

## CELL 10.5 — Translation Cache (FIX F from v15, kept)

In [ ]:
import hashlib
import pickle as _pickle

_TRANSLATION_CACHE      : dict = {}
_TRANSLATION_CACHE_PATH = os.path.join(_WORK_ROOT, 'translation_cache.pkl')

def _load_translation_cache():
    global _TRANSLATION_CACHE
    if os.path.exists(_TRANSLATION_CACHE_PATH):
        try:
            with open(_TRANSLATION_CACHE_PATH, 'rb') as f:
                _TRANSLATION_CACHE = _pickle.load(f)
            print(f'✅ Translation cache loaded: {len(_TRANSLATION_CACHE)} entries')
        except Exception:
            _TRANSLATION_CACHE = {}
            print('⚠️  Translation cache corrupted — starting fresh')
    else:
        print('   Translation cache: empty (first run)')

def _save_translation_cache():
    try:
        with open(_TRANSLATION_CACHE_PATH, 'wb') as f:
            _pickle.dump(_TRANSLATION_CACHE, f)
    except Exception as e:
        print(f'⚠️  Failed to save translation cache: {e}')

_load_translation_cache()

_german_to_english_strict_original = german_to_english_strict

def german_to_english_strict(german_text: str, max_new_tokens: int = 150) -> str:
    if not german_text or not german_text.strip(): return ''
    cache_key = hashlib.md5(german_text.strip().encode('utf-8')).hexdigest()
    if cache_key in _TRANSLATION_CACHE: return _TRANSLATION_CACHE[cache_key]
    result = _german_to_english_strict_original(german_text, max_new_tokens=max_new_tokens)
    _TRANSLATION_CACHE[cache_key] = result
    _save_translation_cache()
    return result

print('✅ Translation cache layer active')
print(f'   Cache size: {len(_TRANSLATION_CACHE)} entries')

## CELL 11 — Path Handlers v16 (German-native, Dict-LoRA enriched)

In [ ]:
def sentence_path(codes, rag_results, meanings, include_english: bool = False):
    """
    v16: All RAG-successful inputs go through here.
    1. Get German from RAG corpus (primary) OR Dict-LoRA enrichment
    2. German IS the output — no mandatory German→English translation
    3. English is generated only if include_english=True (saves ~35s per call)
    4. Arabic from German source directly
    """
    assembled_words = assemble_words(codes, use_tla=True)
    query_orig      = ' '.join(assembled_words)
    query_norm      = normalize_tla(query_orig)

    # Step 1: Get German (from RAG or Dict-LoRA or LLM)
    german = translate_with_rag_prompt(
        query_orig, query_norm, rag_results, meanings,
        assembled_words=assembled_words
    )

    if not german or set(german.strip()) <= set('. -') or len(german.strip()) < 3:
        # Dict-LoRA enrichment as last resort before giving up
        lora_meanings = enrich_meanings_with_lora(assembled_words)
        meaning_hint  = ', '.join(v for v in lora_meanings.values() if v)
        german = meaning_hint or ', '.join(m for m in meanings if m) or query_orig

    # Step 2: English ONLY if requested (secondary output)
    english = ''
    if include_english:
        english = fix_grammar(german_to_english_strict(german))
        if not english or set(english.strip()) <= set('. -'):
            english = german  # Show German as fallback if EN translation fails

    sentiment = get_sentiment(german)  # v16: sentiment on German text
    intention = get_intention(german)
    ar_result = translate_arabic_verified(german)  # v16: German → Arabic
    best      = rag_results[0]

    return {
        'path'             : 'sentence',
        'german'           : german,      # PRIMARY
        'english'          : english,     # SECONDARY (empty unless requested)
        'arabic'           : ar_result['arabic'],
        'arabic_verified'  : ar_result['verified'],
        'arabic_confidence': ar_result['confidence'],
        'sentiment'        : sentiment,
        'intention'        : intention,
        'rag_score'        : best['score'],
        'rag_source'       : best['transcription'],
        'rag_results'      : rag_results,
        'assembled_words'  : assembled_words,
        'german_source'    : german,
    }

print('✅ Path handlers ready (v16 — German-native, Dict-LoRA enriched)')

## CELL 12 — Full Pipeline v16
### Input: Gardiner codes (raw, may include unknown/OCR-broken codes)
### Output: German (primary), Arabic (secondary), English (optional)
### Nearest-valid code snapping → Dict-LoRA word lookup → RAG retrieval → German output

In [ ]:
def full_pipeline(gardiner_input: str, verbose: bool = True,
                   include_english: bool = False) -> dict:
    t0    = time.time()

    # ── Step 1: Parse + Nearest-valid code snapping (NEW v16) ────────────────
    raw_codes = gardiner_input.strip().upper().split()
    codes, snap_log = normalize_codes(raw_codes)

    if verbose and snap_log:
        print(f'⚠️  Code snapping: {snap_log}')

    # ── Step 2: Assemble words from phonetics ────────────────────────────────
    assembled_words = assemble_words(codes, use_tla=True)
    tla_query       = ' '.join(assembled_words)
    tla_norm        = normalize_tla(tla_query)
    meanings        = get_meanings(codes)

    if verbose:
        print(f'Input codes   : {raw_codes}')
        if snap_log: print(f'After snapping: {codes}')
        print(f'Raw + det     : {assemble_raw_with_det(codes)}')
        print(f'Assembled     : {assembled_words}')
        print(f'TLA query     : {tla_query}')
        print(f'TLA norm      : {tla_norm}')

    # ── Step 3: Word Assembly (Trie + DP + fuzzy) ────────────────────────────
    wa = word_assembly_result(assembled_words)

    if verbose:
        print(f'Word Assembly : {wa["best_segmentation"]}')
        print(f'Assembly Score: {wa["best_score"]}')
        print(f'Assembly types: {[m["type"] for m in wa["best_metadata"]]}')
        if wa['alternatives']: print(f'Alternatives  : {wa["alternatives"][:2]}')

    # ── Step 4: Dict-LoRA word enrichment (NEW v16) ──────────────────────────
    lora_meanings = enrich_meanings_with_lora(wa['best_segmentation'])
    if verbose:
        non_empty_lora = {w: m for w, m in lora_meanings.items() if m}
        if non_empty_lora: print(f'Dict-LoRA     : {non_empty_lora}')

    # ── Step 5: Multi-query hybrid RAG search ────────────────────────────────
    rag_results = hybrid_search(
        tla_query, top_k=TOP_K_RAG,
        tla_tokens=assembled_words, codes=codes,
    )
    best_score = rag_results[0]['score'] if rag_results else 0.0

    if verbose:
        print(f'RAG best score: {best_score:.4f} → {interpret_rag_score(best_score)}')
        print(f'Threshold     : {SIMILARITY_THRESHOLD:.4f}')
        if rag_results:
            top = rag_results[0]
            print(f'Top RAG hit   : [{top["transcription"]}] → {top["translation"]}')

    # ── Step 6: Route to sentence_path or fallback ──────────────────────────
    if best_score >= SIMILARITY_THRESHOLD:
        if verbose: print('Path: SENTENCE  ← RAG retrieval succeeded')
        result = sentence_path(codes, rag_results, meanings, include_english=include_english)
    else:
        if verbose: print('Path: FALLBACK  ← below threshold, using Dict-LoRA + LLM')

        # v16 fallback: Dict-LoRA first, then Gardiner meanings, then LLM
        lora_non_empty = [v for v in lora_meanings.values() if v]
        word_meanings  = [m for m in meanings if m]
        per_word_meta  = [m['meaning'] for m in wa['best_metadata'] if m.get('meaning')]

        if lora_non_empty:
            # Dict-LoRA found meanings — use them as German phrase
            german  = ', '.join(lora_non_empty)
        elif per_word_meta:
            german = ' '.join(per_word_meta)
        else:
            best_german = _get_best_german_from_rag(rag_results)
            if best_german:
                german = best_german
            elif word_meanings:
                german = f'Altägyptische Zeichen: {chr(44).join(word_meanings)}.'
            else:
                german = tla_query

        english = ''
        if include_english:
            english = fix_grammar(german_to_english_strict(german))
            if not english or set(english.strip()) <= set('. -'):
                english = german

        ar_result = translate_arabic_verified(german)
        result = {
            'path'             : 'fallback',
            'german'           : german,
            'english'          : english,
            'arabic'           : ar_result['arabic'],
            'arabic_verified'  : ar_result['verified'],
            'arabic_confidence': ar_result['confidence'],
            'assembled_words'  : assembled_words,
            'sentiment'        : get_sentiment(german),
            'intention'        : get_intention(german),
            'rag_score'        : best_score,
            'rag_source'       : rag_results[0]['transcription'] if rag_results else '',
            'rag_results'      : rag_results,
        }

    result.update({
        'input'            : gardiner_input,
        'codes_raw'        : raw_codes,
        'codes_snapped'    : codes,
        'snap_log'         : snap_log,
        'tla_phonemes'     : tla_query,
        'tla_normalised'   : tla_norm,
        'meanings'         : meanings,
        'assembly_metadata': wa['best_metadata'],
        'assembled_words'  : result.get('assembled_words', assembled_words),
        'assembly_alts'    : wa['alternatives'],
        'assembly_score'   : wa['best_score'],
        'lora_meanings'    : lora_meanings,
        'processing_time'  : round(time.time() - t0, 2),
    })

    if verbose:
        print('=' * 60)
        print(f'PATH          : {result["path"].upper()}')
        print(f'Assembled     : {" ".join(result.get("assembled_words", []))}')
        print(f'TLA norm      : {tla_norm}')
        print(f'German (RAG)  : {result["german"]}')
        if result.get('english'):
            print(f'English       : {result["english"]}')
        print(f'Arabic        : {result["arabic"]}')
        ar_ver = result.get('arabic_verified')
        if ar_ver is not None:
            print(f'Arabic check  : {"verified" if ar_ver else "low confidence"} ({result.get("arabic_confidence",0):.0%})')
        s = result.get('sentiment')
        if s: print(f'Sentiment     : {s["label"]} ({s["confidence"]:.0%})')
        i = result.get('intention')
        if i: print(f'Intention     : {i["intention_en"]} / {i["intention_ar"]}')
        rag_s = result['rag_score']
        print(f'RAG Score     : {rag_s:.4f}  →  {interpret_rag_score(rag_s)}')
        print(f'Time          : {result["processing_time"]}s')
        print('=' * 60)
    return result


# ── SMOKE TEST v16 ───────────────────────────────────────────────────────────
print('\n=== SMOKE TEST v16 ===')
print('\nTest 1: Known sequence — S42 D2 F34')
print('Expected German: Leiter der Mitte')
result1 = full_pipeline('S42 D2 F34')

print('\nTest 2: Regression — injury for Horus')
result2 = full_pipeline('S29 M17 G1 V13 D57 N35 S29 M17 D36 V13 D57 G5')

print('\nTest 3: Code snapping — OCR-broken codes with leading zeros')
print('A01 G05 N35  →  should snap to A1 G5 N35')
result3 = full_pipeline('A01 G05 N35')
print(f'Snap log: {result3["snap_log"]}')

## CELL 13 — Full Evaluation Suite v16 (German-native metrics)

In [ ]:
import nltk
from nltk.translate.bleu_score   import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score                 import rouge_scorer
from sacrebleu.metrics           import CHRF
import warnings; warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('omw-1.4', quiet=True)

def calc_bleu(reference, hypothesis):
    ref_tok = reference.lower().split()
    hyp_tok = hypothesis.lower().split()
    if not ref_tok or not hyp_tok: return 0.0
    smoother = SmoothingFunction().method1
    max_n    = min(len(ref_tok), len(hyp_tok), 4)
    weights  = {1:(1,0,0,0), 2:(.5,.5,0,0), 3:(.33,.33,.34,0)}.get(max_n, (.25,.25,.25,.25))
    return sentence_bleu([ref_tok], hyp_tok, weights=weights, smoothing_function=smoother) * 100

def calc_rouge(reference, hypothesis):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    scores = scorer.score(reference.lower(), hypothesis.lower())
    return {k: v.fmeasure * 100 for k, v in scores.items()}

def calc_meteor(reference, hypothesis):
    try: return meteor_score([reference.lower().split()], hypothesis.lower().split()) * 100
    except: return 0.0

def calc_chrf(reference, hypothesis):
    return CHRF().sentence_score(hypothesis, [reference]).score

def calc_exact_match(reference, hypothesis):
    return 100.0 if reference.lower().strip() == hypothesis.lower().strip() else 0.0

def calc_word_overlap(reference, hypothesis):
    ref_words = set(reference.lower().split())
    hyp_words = set(hypothesis.lower().split())
    if not ref_words: return 0.0
    return len(ref_words & hyp_words) / len(ref_words) * 100

def calc_recall_at_k(reference_german, retrieved, k_values=[1,3,5,10,20]):
    ref_norm = normalize_tla(reference_german).lower()
    return {
        f'recall@{k}': 100.0 if any(
            fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70
            for r in retrieved[:k]
        ) else 0.0
        for k in k_values
    }

def calc_mrr(reference_german, retrieved):
    ref_norm = normalize_tla(reference_german).lower()
    for rank, r in enumerate(retrieved, 1):
        if fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70:
            return 100.0 / rank
    return 0.0

def evaluate_pipeline(gardiner_input, reference_german='', reference_english='',
                      include_english: bool = True):
    """
    v16: Primary reference is German (native corpus language).
    English metrics computed only if reference_english is given.
    """
    result = full_pipeline(gardiner_input, verbose=False, include_english=include_english)
    hyp_de = result.get('german', '')
    hyp_en = result.get('english', '')

    # German metrics (PRIMARY)
    rouge_de = calc_rouge(reference_german, hyp_de) if reference_german else {}
    recall   = calc_recall_at_k(reference_german, result.get('rag_results', [])) if reference_german else {}
    mrr      = calc_mrr(reference_german, result.get('rag_results', [])) if reference_german else 0.0

    # English metrics (SECONDARY, only if reference_english given)
    rouge_en = calc_rouge(reference_english, hyp_en) if (reference_english and hyp_en) else {}

    return {
        **result,
        'reference_german' : reference_german,
        'reference_english': reference_english,
        # German metrics
        'bleu_de'          : calc_bleu(reference_german, hyp_de)      if reference_german else 0.0,
        'rouge1_de'        : rouge_de.get('rouge1', 0.0),
        'rouge2_de'        : rouge_de.get('rouge2', 0.0),
        'rougeL_de'        : rouge_de.get('rougeL', 0.0),
        'meteor_de'        : calc_meteor(reference_german, hyp_de)    if reference_german else 0.0,
        'chrf_de'          : calc_chrf(reference_german, hyp_de)       if reference_german else 0.0,
        'exact_match_de'   : calc_exact_match(reference_german, hyp_de) if reference_german else 0.0,
        'word_overlap_de'  : calc_word_overlap(reference_german, hyp_de) if reference_german else 0.0,
        # English metrics (secondary)
        'bleu_en'          : calc_bleu(reference_english, hyp_en)     if (reference_english and hyp_en) else 0.0,
        'rouge1_en'        : rouge_en.get('rouge1', 0.0),
        # Retrieval
        **recall,
        'mrr'              : mrr,
        'avg_retrieval_score': np.mean(list(recall.values())) if recall else 0.0,
    }

print('✅ Evaluation suite ready (v16 — German-native metrics)')

## CELL 14 — RAG vs Dict-LoRA vs LLM-Only Comparison (v16)

In [ ]:
def translate_llm_only(query_original: str) -> str:
    """Direct Seed-X translation WITHOUT RAG or Dict-LoRA."""
    prompt = (
        'Du bist ein Experte für altägyptische Grammatik.\n\n'
        'Übersetze die folgende ältere ägyptische Transliteration ins Deutsche.\n'
        'Gib NUR die deutsche Übersetzung aus.\n\n'
        f'Ägyptische Transliteration:\n{query_original}\n\n'
        'Deutsche Übersetzung:\n'
    )
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=150, num_beams=1, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    return seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()

def translate_lora_only(assembled_words: list) -> str:
    """NEW v16: Dict-LoRA only — no RAG, no Seed-X."""
    lora_meanings = enrich_meanings_with_lora(assembled_words)
    non_empty = [v for v in lora_meanings.values() if v]
    return ', '.join(non_empty) if non_empty else ''

def run_full_comparison(num_samples: int = None):
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available. Run CELL 6 first.')
        return None, None, None
    test_df = df_test_global if num_samples is None else df_test_global.head(num_samples)
    print('=' * 70)
    print(f'🆚 RAG vs Dict-LoRA vs LLM-ONLY ({len(test_df)} samples)')
    print('=' * 70)
    rag_results_list  = []
    llm_results_list  = []
    lora_results_list = []

    for idx in range(len(test_df)):
        row          = test_df.iloc[idx]
        query_orig   = row['transliteration']
        reference_de = row['translation']   # German — native

        # RAG pipeline
        rag = evaluate_pipeline(query_orig, reference_german=reference_de)
        rag_results_list.append(rag)

        # LLM-only (German output)
        llm_german = translate_llm_only(query_orig)
        rouge_llm  = calc_rouge(reference_de, llm_german)
        llm_results_list.append({
            'bleu_de': calc_bleu(reference_de, llm_german),
            'rouge1_de': rouge_llm.get('rouge1', 0.0),
            'rouge2_de': rouge_llm.get('rouge2', 0.0),
            'rougeL_de': rouge_llm.get('rougeL', 0.0),
            'meteor_de': calc_meteor(reference_de, llm_german),
            'chrf_de'  : calc_chrf(reference_de, llm_german),
            'exact_match_de': calc_exact_match(reference_de, llm_german),
            'word_overlap_de': calc_word_overlap(reference_de, llm_german),
        })

        # Dict-LoRA only
        assembled = assemble_words(query_orig.strip().upper().split(), use_tla=True)
        lora_de   = translate_lora_only(assembled)
        rouge_lora = calc_rouge(reference_de, lora_de) if lora_de else {}
        lora_results_list.append({
            'bleu_de': calc_bleu(reference_de, lora_de) if lora_de else 0.0,
            'rouge1_de': rouge_lora.get('rouge1', 0.0),
            'chrf_de'  : calc_chrf(reference_de, lora_de) if lora_de else 0.0,
        })

        if (idx + 1) % 10 == 0:
            print(f'  Processed {idx + 1}/{len(test_df)} samples...')

    df_rag  = pd.DataFrame(rag_results_list)
    df_llm  = pd.DataFrame(llm_results_list)
    df_lora = pd.DataFrame(lora_results_list)

    metric_cols = ['bleu_de','rouge1_de','rouge2_de','rougeL_de','meteor_de','chrf_de',
                   'exact_match_de','word_overlap_de']

    print('\n📊 COMPARISON SUMMARY (German metrics)')
    print(f'{"Metric":<22} {"RAG":>8} {"Dict-LoRA":>10} {"LLM-Only":>10}')
    print('-' * 55)
    for m in metric_cols:
        rag_m  = df_rag[m].mean()  if m in df_rag.columns  else 0.0
        lora_m = df_lora[m].mean() if m in df_lora.columns else 0.0
        llm_m  = df_llm[m].mean()  if m in df_llm.columns  else 0.0
        print(f'{m:<22} {rag_m:>8.2f} {lora_m:>10.2f} {llm_m:>10.2f}')

    df_rag.to_csv(os.path.join(_WORK_ROOT, 'rag_eval_results_v16.csv'), index=False)
    df_llm.to_csv(os.path.join(_WORK_ROOT, 'llm_eval_results_v16.csv'), index=False)
    df_lora.to_csv(os.path.join(_WORK_ROOT, 'lora_eval_results_v16.csv'), index=False)
    print('\n💾 Results saved')
    return df_rag, df_llm, df_lora

# df_rag, df_llm, df_lora = run_full_comparison(num_samples=50)

## CELL 15 — K-Value Optimization (kept from v15, now on German metrics)

In [ ]:
def run_k_optimization(num_samples: int = 10,
                        k_min: int = 5, k_max: int = 205, k_step: int = 5):
    global TOP_K_RAG
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available.'); return None
    k_values = list(range(k_min, k_max, k_step))
    test_df  = df_test_global.head(num_samples)
    test_samples = []
    for idx in range(len(test_df)):
        q_orig = test_df.iloc[idx]['transliteration']
        ref_de = test_df.iloc[idx]['translation']       # German — primary
        q_norm = normalize_tla(q_orig)
        test_samples.append({'q_orig': q_orig, 'q_norm': q_norm, 'ref_de': ref_de})
    k_results = []
    for k in k_values:
        TOP_K_RAG = k
        bleu_scores, recall_scores = [], []
        for s in test_samples:
            rag    = hybrid_search(s['q_orig'], top_k=k)
            recall = calc_recall_at_k(s['ref_de'], rag, k_values=[k])
            german = _get_best_german_from_rag(rag) if rag else ''
            bleu_scores.append(calc_bleu(s['ref_de'], german))
            recall_scores.append(recall.get(f'recall@{k}', 0.0))
        avg_bleu   = np.mean(bleu_scores)
        avg_recall = np.mean(recall_scores)
        composite  = 0.5 * avg_bleu + 0.5 * avg_recall
        k_results.append({'k': k, 'bleu_de': avg_bleu, f'recall@{k}': avg_recall, 'composite': composite})
        print(f'  K={k:3d} | BLEU(de)={avg_bleu:5.2f} | Recall={avg_recall:5.2f} | Composite={composite:5.2f}')
    df_k     = pd.DataFrame(k_results)
    best_row = df_k.loc[df_k['composite'].idxmax()]
    TOP_K_RAG = int(best_row['k'])
    print(f'\n🏆 Optimal K = {TOP_K_RAG} (composite = {best_row["composite"]:.2f})')
    df_k.to_csv(os.path.join(_WORK_ROOT, 'k_optimization_results_v16.csv'), index=False)
    return df_k

# df_k_results = run_k_optimization(num_samples=10)

## CELL 16 — Kill Old Flask Process

In [ ]:
import os
os.system('kill -9 $(lsof -t -i:5006) 2>/dev/null')
time.sleep(2)
print('✅ Port 5006 cleared')

## CELL 17 — Flask API v16 (German-native endpoints)

In [ ]:
import threading
from pyngrok import ngrok, conf
from flask import Flask, request, jsonify
from flask_cors import CORS

ngrok.kill(); time.sleep(2)

NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')
if not NGROK_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_AUTH_TOKEN')
    except Exception: pass
if not NGROK_TOKEN:
    print('⚠️  NGROK_AUTH_TOKEN not set. Add it via: os.environ["NGROK_AUTH_TOKEN"] = "your_token"')
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)
CORS(app)

ERROR_CODES = {
    'MISSING_INPUT' : ('400', 'Required field is missing'),
    'INVALID_INPUT' : ('400', 'Input is malformed or empty'),
    'PIPELINE_ERROR': ('500', 'Internal pipeline failure'),
    'MODEL_ERROR'   : ('503', 'Language model unavailable'),
}

def _api_error(code_key, detail='', status=None):
    code, msg = ERROR_CODES.get(code_key, ('500', 'Unknown error'))
    return jsonify({'success': False,
                    'error': {'code': code_key, 'message': msg, 'detail': detail}}), status or int(code)

def _run_pipeline_safe(gardiner_str, include_english=False):
    last_exc = None
    for attempt in range(2):
        try:
            return full_pipeline(gardiner_str, verbose=False, include_english=include_english), None
        except RuntimeError as e:
            last_exc = e; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            return None, str(e)
    return None, str(last_exc)


@app.route('/api/decipher', methods=['POST'])
def decipher():
    data  = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    codes = data.get('codes', [])
    if not codes: return _api_error('MISSING_INPUT', '"codes" list is required')
    if not isinstance(codes, list) or not all(isinstance(c, str) for c in codes):
        return _api_error('INVALID_INPUT', '"codes" must be a list of strings')
    include_en   = bool(data.get('include_english', False))  # v16: opt-in
    gardiner_str = ' '.join(codes).upper()
    result, err  = _run_pipeline_safe(gardiner_str, include_english=include_en)
    if err: return _api_error('PIPELINE_ERROR', err)

    # Build per-sign breakdown (after snapping)
    code_list  = result['codes_snapped']
    tla_tokens = result['tla_phonemes'].split()
    meanings   = result['meanings']
    per_sign   = []
    for i, c in enumerate(code_list):
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        per_sign.append([c, info.get('unicode', ''),
                          tla_tokens[i] if i < len(tla_tokens) else '',
                          meanings[i]   if i < len(meanings)   else ''])
    glyphs_str = ''.join(GARDINER_MAP.get(c.strip().lower(), {}).get('unicode', '')
                          for c in code_list)
    intent = result.get('intention', {})
    sent   = result.get('sentiment', {})
    rag_s  = result['rag_score']
    return jsonify({'success': True, 'data': {
        'path'                    : result['path'],
        'glyphs'                  : glyphs_str,
        'per_sign'                : per_sign,
        'snap_log'                : result.get('snap_log', []),   # NEW v16
        'tla_norm'                : result['tla_normalised'],
        'german'                  : result['german'],              # PRIMARY (v16)
        'english'                 : result.get('english', ''),    # SECONDARY (v16)
        'arabic'                  : result['arabic'],
        'arabic_verified'         : result.get('arabic_verified'),
        'arabic_confidence'       : result.get('arabic_confidence'),
        'sentiment'               : sent.get('label', 'neutral').lower(),
        'sent_score'              : f"{sent.get('confidence', 0):.0%}",
        'intention_en'            : intent.get('intention_en', ''),
        'intention_ar'            : intent.get('intention_ar', ''),
        'intent_confidence'       : intent.get('confidence'),
        'rag_score'               : rag_s,
        'rag_score_max'           : RRF_SCORE_MAX,
        'rag_score_interpretation': interpret_rag_score(rag_s),
        'assembled_words'         : result.get('assembled_words', []),
        'assembly_metadata'       : result.get('assembly_metadata', []),
        'assembly_alternatives'   : result.get('assembly_alts', []),
        'assembly_score'          : result.get('assembly_score'),
        'lora_meanings'           : result.get('lora_meanings', {}),  # NEW v16
        'german_source'           : result.get('german_source', ''),
        'time'                    : result['processing_time'],
    }}), 200


@app.route('/api/translate', methods=['POST'])
def api_translate():
    data     = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner: return _api_error('MISSING_INPUT', '"gardiner" field is required')
    include_en  = bool(data.get('include_english', False))
    result, err = _run_pipeline_safe(gardiner, include_english=include_en)
    if err: return _api_error('PIPELINE_ERROR', err)
    rag_s = result['rag_score']
    return jsonify({'success': True,
        'input'                   : result['input'],
        'path'                    : result['path'],
        'codes_snapped'           : result.get('codes_snapped', []),
        'snap_log'                : result.get('snap_log', []),
        'tla_phonemes'            : result['tla_phonemes'],
        'tla_normalised'          : result['tla_normalised'],
        'meanings'                : result['meanings'],
        'german'                  : result['german'],         # PRIMARY
        'english'                 : result.get('english',''), # SECONDARY
        'arabic'                  : result['arabic'],
        'arabic_verified'         : result.get('arabic_verified'),
        'sentiment'               : result.get('sentiment'),
        'intention'               : result.get('intention'),
        'lora_meanings'           : result.get('lora_meanings', {}),
        'rag_score'               : rag_s,
        'rag_score_max'           : RRF_SCORE_MAX,
        'rag_score_interpretation': interpret_rag_score(rag_s),
        'rag_results'             : result.get('rag_results', [])[:3],
        'german_source'           : result.get('german_source', ''),
        'time'                    : result['processing_time'],
    }), 200


@app.route('/api/evaluate', methods=['POST'])
def api_evaluate():
    data     = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner: return _api_error('MISSING_INPUT', '"gardiner" field is required')
    ref_de = data.get('reference_german',  '')   # PRIMARY
    ref_en = data.get('reference_english', '')   # SECONDARY
    try:
        result = evaluate_pipeline(gardiner, reference_german=ref_de,
                                   reference_english=ref_en, include_english=bool(ref_en))
    except Exception as e:
        return _api_error('PIPELINE_ERROR', str(e))
    eval_metrics = {k: result[k] for k in [
        'bleu_de','rouge1_de','rouge2_de','rougeL_de','meteor_de','chrf_de',
        'exact_match_de','word_overlap_de',
        'bleu_en','rouge1_en',
        'recall@1','recall@3','recall@5','recall@10','recall@20',
        'mrr','avg_retrieval_score',
    ] if k in result}
    return jsonify({'success': True, 'input': gardiner,
                    'german': result['german'], 'english': result.get('english',''),
                    'metrics': eval_metrics}), 200


@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({
        'status'              : 'ready',
        'version'             : 'v16-German-Native-DictLoRA',
        'environment'         : ENV,
        'model'               : 'Seed-X-PPO-7B',
        'primary_language'    : 'German',
        'secondary_language'  : 'English (on request)',
        'dict_lora_ready'     : _LORA_READY,
        'lora_base_model'     : LORA_BASE_MODEL_ID,
        'signs_loaded'        : len(GARDINER_MAP),
        'valid_codes'         : len(VALID_CODES),
        'code_snapping'       : True,
        'faiss_vectors'       : faiss_index.ntotal if faiss_index else 0,
        'test_set_rows'       : len(df_test_global) if df_test_global is not None else 0,
        'intention_classes'   : len(INTENTION_MAP),
        'prompt_token_limit'  : MAX_PROMPT_TOKENS,
        'rag_k'               : RRF_K,
        'rag_alpha'           : HYBRID_ALPHA,
        'rag_score_max'       : RRF_SCORE_MAX,
        'similarity_threshold': SIMILARITY_THRESHOLD,
    }), 200


@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': 'v16-German-Native-DictLoRA'}), 200


@app.route('/api/examples', methods=['GET'])
def examples():
    # v16: examples include include_english flag
    return jsonify({
        'simple_word'   : {'codes': ['O1'],                'description': 'House', 'include_english': False},
        'monument'      : {'codes': ['G17', 'N35', 'D21'], 'description': 'Monument'},
        'sun_disk'      : {'codes': ['N5'],                'description': 'Sonne / Sun'},
        'son_of_ra'     : {'codes': ['O4', 'N5'],          'description': 'Sohn des Ra / Son of Ra'},
        'royal_offering': {'codes': ['R4', 'X8', 'A42'],   'description': 'Königsopfer / Royal offering'},
        'horus_injury'  : {'codes': ['S29','M17','G1','V13','D57','N35','S29','M17','D36','V13','D57','G5'],
                           'description': 'Verletzung für den der Horus verletzt'},
        'snapped_codes' : {'codes': ['A01','G05','N35'],   'description': 'OCR-broken codes — auto-snapped'},
    }), 200


def run_server():
    app.run(host='0.0.0.0', port=5010, use_reloader=False, debug=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

tunnel     = ngrok.connect(5010)
PUBLIC_URL = tunnel.public_url

print('=' * 65)
print('🏛️  HIEROGLYPH NLP PIPELINE v16 — GERMAN-NATIVE + DICT-LORA')
print(f'🌐  Public URL:\n\n    {PUBLIC_URL}\n')
print('📡  Endpoints:')
print(f'      POST {PUBLIC_URL}/api/decipher      (codes list + optional include_english)')
print(f'      POST {PUBLIC_URL}/api/translate     (gardiner string + optional include_english)')
print(f'      POST {PUBLIC_URL}/api/evaluate      (+ reference_german, reference_english)')
print(f'      GET  {PUBLIC_URL}/api/status')
print(f'      GET  {PUBLIC_URL}/api/health')
print(f'      GET  {PUBLIC_URL}/api/examples')
print('\n🆕 v16 Changes:')
print('   • German is PRIMARY output (corpus language, no double-translation)')
print('   • Dict-LoRA: fine-tuned on BBAW vocabulary — fills gaps CSV misses')
print('   • Nearest-valid code snapping — fixes OCR/typo Gardiner codes')
print('   • include_english=true flag to optionally get English too')
print('=' * 65)